# ML-04 — Search Intelligence Data Contract

## 1. Unit of analysis + time window

**Lane 2: Refresh / Content Opportunity Scoring.** One row in the daily warehouse table represents one **content page × client × report date** observation. For this Week 3 contract, I will use the **March 2026** partition as the development/verification month. I will not use the June 2026 `_sample` for label development because it is the final month and should remain a sealed outcome/test window.

In [9]:
# Query 1 — verify the stated grain for the March 2026 slice.
# Expected result: zero rows means no duplicate client × content × date records.

import duckdb
import os

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

grain_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print('Duplicate grain rows:', len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

- **Features:** earlier-window search/analytics measurements that are available before the ranking decision, such as prior impressions, clicks, sessions, and position summaries.
- **Label / proxy:** a decline outcome/proxy defined from a later outcome window. A label-derived field is never used as a feature.
- **Context:** `client_hash_id` and `content_hash_id` are identifiers used for joining, grouping, and grouped validation; they are not model features.
- **Excluded:** future-window measurements, `trend_direction`/`trend_pct` when they are used to derive the target, and product/private decision flags. They would leak information or encode the decision we are trying to support.

**Output:** a ranked queue of content pages for human review, with the priority driven by observable evidence.

**Availability rule:** analytics features are only considered available when the corresponding `ga4_data_available` flag is true; zero-filled GA4 values before availability are not treated as genuine zero engagement.

In [10]:
# Query 2 — verify the March 2026 slice row count and date span.

slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()
slice_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 3. Verify it with queries + build five features

The three verification queries below check the grain, March slice size/date span, and GA4 availability. After those checks, the five-feature frame uses only information intended to be available before a later decision window.

In [11]:
# Query 3 — verify GA4 availability using IS TRUE, as required by the contract.

availability_check = con.sql(f"""
SELECT COUNT(*) AS rows_with_ga4_available
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND ga4_data_available IS TRUE
""").df()
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4_available
0,413966


### Five-feature frame

For the five features, I use March 2026 page-level observations and aggregate the daily data within the month. The feature definitions are written so their availability is explicit:

1. **impressions_90d** — knowable at the decision moment because it summarizes search impressions observed before that decision.
2. **clicks_90d** — knowable at the decision moment because it summarizes clicks already observed before the decision.
3. **sessions_90d** — knowable at the decision moment because it summarizes observed analytics sessions, subject to the GA4 availability flag.
4. **avg_position** — knowable at the decision moment because it summarizes observed search position before the decision.
5. **content_age_days** — knowable at the decision moment because page age is determined from the content's existing history.

The final model should enforce a strict feature window before the outcome window; these are a contract-level feature sketch, not proof of causal usefulness.

In [12]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_march,

    SUM(gsc_clicks) AS clicks_march,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE 0
        END
    ) AS sessions_march,

    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,

    COUNT(DISTINCT report_date) AS days_observed

FROM {DAILY}

WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature-frame shape:", features.shape)
display(features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame shape: (331437, 7)


,client_hash_id,content_hash_id,impressions_march,clicks_march,sessions_march,avg_position_march,days_observed
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.0,4.394234,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.0,7.842593,31
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,4.0,8.454069,31
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,9.0,6.320337,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.0,4.459107,31
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,0.0,16.306140,31
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,1.0,7.046534,31
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,0.0,19.563725,31
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,8.0,4.950311,31
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,1.0,52.127896,31


### 4. The trap: deliberate label leakage

I will deliberately add a label-derived column to a small quick classification experiment. Because the feature contains the answer, a very high score is expected and is **not** evidence of a useful model. After demonstrating the trap, the leaked column is removed and the honest feature set is retained.

The important lesson is that a feature must be knowable before the prediction/review moment and must not be computed from the label or its outcome window.

In [13]:
# Deliberate leakage experiment

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

leak_df = features.copy()

# Create a simple proxy target for this demonstration.
leak_df["decline_proxy"] = (
    leak_df["impressions_march"] < leak_df["impressions_march"].median()
).astype(int)

# DELIBERATE LEAK:
# This column contains the target itself.
leak_df["LEAK_LABEL"] = leak_df["decline_proxy"]

feature_cols_leaky = [
    "impressions_march",
    "clicks_march",
    "sessions_march",
    "avg_position_march",
    "days_observed",
    "LEAK_LABEL"
]

model_data = leak_df.dropna(subset=feature_cols_leaky)

X_train, X_test, y_train, y_test = train_test_split(
    model_data[feature_cols_leaky],
    model_data["decline_proxy"],
    test_size=0.25,
    random_state=42,
    stratify=model_data["decline_proxy"]
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

print(
    f"Leaky quick accuracy: "
    f"{accuracy_score(y_test, leaky_pred):.3f}"
)

# Remove the deliberately leaked column.
features_honest = features.copy()

print("LEAK_LABEL present after cleanup:",
      "LEAK_LABEL" in features_honest.columns)

print("Honest feature columns:")
print(list(features_honest.columns))

Leaky quick accuracy: 1.000
LEAK_LABEL present after cleanup: False
Honest feature columns:
['client_hash_id', 'content_hash_id', 'impressions_march', 'clicks_march', 'sessions_march', 'avg_position_march', 'days_observed']


## Data limits

One important limitation is that the warehouse is an **unbalanced panel**: clients have different history depths. Therefore, a March row does not imply that every client has the same amount of usable history before March. GA4 availability also differs by client, so missing/zero-filled analytics values cannot be interpreted as comparable across all clients without using the availability flag.

The March slice is useful for development and verification, but it should not be treated as a final test. June 2026 is the final month and should remain sealed for final evaluation.

## Self-check

- [x] Unit of analysis and March 2026 development window stated.
- [x] Feature, label/proxy, context, and excluded fields classified.
- [x] Exactly three verification queries are provided: grain, row count/date span, and `IS TRUE` availability.
- [x] Five-feature frame provided with an availability explanation for each feature.
- [x] Label leakage is deliberately demonstrated and the leaked column is removed.
- [x] A concrete limitation of the slice is documented.
- [ ] Run all cells in Colab so the three warehouse query outputs and leakage result are genuinely recorded before committing.
